In [1]:
import pandas as pd
import requests
import os
import re
from urllib.parse import urlparse

In [7]:
def sanitize_filename(filename):
    """Remove invalid characters from filename"""
    return re.sub(r'[<>:"/\\|?*]', '', filename).strip()

def download_image(url, filepath):
    """Download image from URL"""
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        with open(filepath, 'wb') as f:
            f.write(response.content)
        return True
    except:
        return False

# Load data
spectral_df = pd.read_csv('../data/results/spectral_13_reps.csv')
anime_df = pd.read_csv('../data/anime_old.csv')

# Create English name lookup
anime_lookup = anime_df.set_index('MAL_ID')['English name'].to_dict()

# Create output folder
os.makedirs('anime_images', exist_ok=True)

# Download images
total = len(spectral_df)
for i, (_, row) in enumerate(spectral_df.iterrows(), 1):
    try:
        anime_id = row['anime_id']
        cluster = row['cluster_id']
        default_title = row['title']
        image_url = row['main_pic']
        
        # Get English name or use default
        english_name = anime_lookup.get(anime_id, default_title)
        if pd.isna(english_name) or english_name.lower() in ['unknown', 'nan', '']:
            english_name = default_title
        
        # Create filename
        filename = f"Cluster_{cluster}_{sanitize_filename(english_name)}"
        
        # Get file extension from URL
        ext = os.path.splitext(urlparse(image_url).path)[1] or '.jpg'
        filepath = f"anime_images/{filename}{ext}"
        
        # Download if not exists
        if not os.path.exists(filepath):
            if download_image(image_url, filepath):
                print(f"[{i}/{total}] Downloaded: {filename}")
            else:
                print(f"[{i}/{total}] Failed: {filename}")
        else:
            print(f"[{i}/{total}] Exists: {filename}")
            
    except Exception as e:
        print(f"[{i}/{total}] Error processing row: {e}")
        continue

print(f"Done! Images saved in 'anime_images' folder")

[1/130] Exists: Cluster_6_A Silent Voice
[2/130] Exists: Cluster_6_Sword Art Online
[3/130] Exists: Cluster_6_Overlord
[4/130] Exists: Cluster_6_My Hero Academia 4
[5/130] Exists: Cluster_6_Food Wars! The Third PlateTotsuki Train Arc
[6/130] Exists: Cluster_6_JoJo's Bizarre Adventure
[7/130] Exists: Cluster_6_MagiThe Labyrinth of Magic
[8/130] Exists: Cluster_6_Hellsing Ultimate
[9/130] Exists: Cluster_6_Dr. Stone Stone Wars
[10/130] Failed: Cluster_6_Dragon Ball Z
[11/130] Exists: Cluster_2_Ghost in the Shell
[12/130] Exists: Cluster_2_Whisper of the Heart
[13/130] Exists: Cluster_2_Kino's Journey
[14/130] Exists: Cluster_2_Summer Wars
[15/130] Exists: Cluster_2_Tokyo Godfathers
[16/130] Exists: Cluster_2_BerserkThe Golden Age Arc II - The Battle for Doldrey
[17/130] Exists: Cluster_2_The Vision of Escaflowne
[18/130] Exists: Cluster_2_Space Dandy 2nd Season
[19/130] Exists: Cluster_2_Detroit Metal CityThe Animated Series
[20/130] Exists: Cluster_2_Trigun - Badlands Rumble
[21/130] Ex